# Model Training

This notebook is optimized for the final deadline sprint.

It does four things:

1. builds a strict grouped evaluation pipeline using the existing `Region` column,
2. tests a small shortlist of target-specific candidates,
3. freezes a safe manifest from full grouped CV,
4. writes three submission files:
   - **A** = safe anchor,
   - **B** = EC aggressive + DRP safe,
   - **C** = hedge blend.

Notes:
- We assume `Region` already exists in the provided dataset.
- We keep the notebook cell-by-cell and avoid one giant integrated script.
- We clip predictions to nonnegative values before submission.

In [ ]:
import os
import sys
import json
import time
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet

from xgboost import XGBRegressor
from IPython.display import display

## Environment and MLflow

In [ ]:
sys.path.append(os.path.abspath('..'))

ENV = 'local'   # switch to 'snowflake' if needed

if ENV == 'local':
    from src import config_local as config
else:
    from src import config_snowflake as config

mlflow.set_tracking_uri(config.MLFLOW_URI)
mlflow.set_experiment('WaterQuality')

print('MLflow URI:', config.MLFLOW_URI)

## Global config

This cell defines:
- targets,
- split metadata,
- artifact directory,
- hashing helpers,
- submission integrity checks.

In [ ]:
TARGET_COLS = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus'
]

SPLIT_STRATEGY = 'LORO'
GROUP_DEFINITION_VERSION = 'region_v1'
PIPELINE_VERSION = 'deadline_v2_patch'
PREPROCESS_VERSION = 'median_scaler'
ARTIFACT_DIR = '../models/final_deadline'

os.makedirs(ARTIFACT_DIR, exist_ok=True)


def hash_str(s: str) -> str:
    '''
    Create a short stable hash from a string.
    '''
    return hashlib.sha256(s.encode('utf-8')).hexdigest()[:16]


def hash_list(values) -> str:
    '''
    Hash a list of values after converting to strings.
    '''
    return hash_str('||'.join(map(str, values)))


def compute_group_values_hash(groups: pd.Series) -> str:
    '''
    Hash the exact ordered group assignments.
    Useful to ensure runs are truly comparable.
    '''
    return hash_list(groups.fillna('NA').astype(str).tolist())


def compute_feature_set_hash(features: list) -> str:
    '''
    Hash a feature list in sorted form.
    '''
    return hash_list(sorted(features))


def target_key(target_name: str) -> str:
    '''
    Make a target name filename-safe.
    '''
    return target_name.replace(' ', '')


def make_row_id_template(template_df: pd.DataFrame) -> pd.DataFrame:
    '''
    Add an immutable row_id to the submission template
    so row order can be validated before saving.
    '''
    out = template_df.copy()
    out['row_id'] = np.arange(len(out), dtype=int)
    return out


def assert_submission_integrity(sub_df: pd.DataFrame, template_df: pd.DataFrame, target_cols: list):
    '''
    Validate that the submission is structurally safe.
    '''
    if len(sub_df) != len(template_df):
        raise RuntimeError(f'Row count mismatch: sub={len(sub_df)} template={len(template_df)}')

    if 'row_id' not in sub_df.columns or 'row_id' not in template_df.columns:
        raise RuntimeError('row_id missing in submission/template.')

    if sub_df['row_id'].duplicated().any():
        raise RuntimeError('Duplicate row_id in submission.')

    if not sub_df['row_id'].equals(template_df['row_id']):
        raise RuntimeError('row_id order mismatch.')

    if sub_df[target_cols].isnull().any().any():
        raise RuntimeError('NaN found in target predictions.')

    if (sub_df[target_cols] < 0).any().any():
        raise RuntimeError('Negative predictions found.')


print('Global config loaded.')

## Data loading

We load the training data and confirm that the `Region` column already exists.

In [ ]:
df = config.load_data()

if 'Region' not in df.columns:
    raise RuntimeError('Region column is required for LORO but was not found.')

if df['Region'].nunique() < 2:
    raise RuntimeError('Need at least 2 unique regions for grouped CV.')

print('Training shape:', df.shape)
print('\nRegion counts:')
print(df['Region'].value_counts(dropna=False))

## Feature engineering

This notebook only uses the two engineered features that were explicitly confirmed from the winning setup:

- `pop_density_upstream`
- `specific_discharge`

In [ ]:
def safe_divide(num, den, eps=1e-8):
    '''
    Divide safely and return NaN when the denominator is effectively zero.
    '''
    num = pd.Series(num, copy=False).astype(float)
    den = pd.Series(den, copy=False).astype(float)
    out = np.where(np.abs(den) < eps, np.nan, num / den)
    return pd.Series(out, index=num.index)


def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    '''
    Create only the confirmed engineered features from the winning setup.
    '''
    df_eng = data.copy()

    df_eng['pop_density_upstream'] = (
        df_eng['worldpop_mean_1km'] /
        (df_eng['basin_upstream_area_km2'] + 1e-5)
    )

    df_eng['specific_discharge'] = (
        df_eng['river_avg_discharge_cms'] /
        (df_eng['basin_upstream_area_km2'] + 1e-5)
    )

    df_eng[['pop_density_upstream', 'specific_discharge']] = (
        df_eng[['pop_density_upstream', 'specific_discharge']]
        .replace([np.inf, -np.inf], np.nan)
    )

    return df_eng

df = engineer_features(df)

print('Engineered features added:')
print([
    'pop_density_upstream',
    'specific_discharge',
    'aridity_index',
    'wind_exposure',
    'moisture_heat_ratio'
])

## Frozen feature sets

We use two frozen feature sets:

- **A** = stable base set
- **B** = base + empirical interactions

In [ ]:
BASE_FEATURES = [
    'nir', 'green', 'NDMI', 'MNDWI', 'pet', 'elevation_meters',
    'total_precipitation', 'average_wind_speed',
    'soil_phh2o_mean_0_5cm', 'soil_clay_mean_0_5cm', 'soil_sand_mean_0_5cm',
    'soil_silt_mean_0_5cm', 'soil_cec_mean_0_5cm',
    'sanlc2022_impact_1km', 'sanlc2020_impact_1km', 'sanlc_change_2020_2022',
    'worldpop_mean_1km', 'basin_upstream_area_km2', 'river_avg_discharge_cms'
]

ENGINEERED_CONFIRMED = [
    'pop_density_upstream',
    'specific_discharge'
]

FEATURE_SETS = {
    'A': BASE_FEATURES,
    'B': BASE_FEATURES + ENGINEERED_CONFIRMED,
}


def features_for_set(df_local: pd.DataFrame, fs_name: str):
    '''
    Return a strict feature list for a named feature set.
    Fail fast if expected columns are missing.
    '''
    requested = FEATURE_SETS[fs_name]
    missing = [f for f in requested if f not in df_local.columns]
    if missing:
        raise RuntimeError(f'Missing in {fs_name}: {missing}')
    return requested


print('Feature sets ready:')
for k, v in FEATURE_SETS.items():
    print(f'{k}: {len(v)} features')

## Preprocessing

We use median imputation and standard scaling inside the CV pipeline.

In [ ]:
def get_preprocessor(features_used):
    '''
    Build the preprocessing pipeline for numeric features.
    '''
    numeric_pipe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    return ColumnTransformer(
        transformers=[('num', numeric_pipe, features_used)],
        remainder='drop'
    )

## Models and shortlist

This shortlist is deliberately small:
- TA: mostly stable XGB
- EC: linear empirical + one XGB challenger
- DRP: safer linear options + one shallow XGB challenger

In [ ]:
def log_wrap(model):
    '''
    Wrap a regressor with log1p / expm1 target transformation.
    '''
    return TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )


def make_xgb(**overrides):
    '''
    Build a regularized XGBoost regressor for spatially robust behavior.
    '''
    params = {
        'objective': 'reg:squarederror',
        'n_estimators': 300,
        'learning_rate': 0.03,
        'max_depth': 4,
        'min_child_weight': 10,
        'subsample': 0.70,
        'colsample_bytree': 0.70,
        'reg_alpha': 1.0,
        'reg_lambda': 6.0,
        'random_state': 42,
        'n_jobs': -1,
    }
    params.update(overrides)
    return XGBRegressor(**params)


MODEL_BANK = {
    'Ridge_a10_Log': log_wrap(Ridge(alpha=10.0)),
    'Ridge_a30_Log': log_wrap(Ridge(alpha=30.0)),
    'Lasso_a0.0005_Log': log_wrap(Lasso(alpha=0.0005, max_iter=20000)),
    'Lasso_a0.0010_Log': log_wrap(Lasso(alpha=0.0010, max_iter=20000)),
    'Elastic_a0.001_l07_Log': log_wrap(ElasticNet(alpha=0.001, l1_ratio=0.7, max_iter=20000, random_state=42)),
    'XGB_d4_lr003_Log': log_wrap(make_xgb(n_estimators=300, learning_rate=0.03, max_depth=4)),
    'XGB_d3_lr003_Log': log_wrap(make_xgb(n_estimators=220, learning_rate=0.03, max_depth=3)),
}

TARGET_SWEEP = {
    'Total Alkalinity': [
        ('A', 'XGB_d4_lr003_Log'),
        ('B', 'XGB_d4_lr003_Log'),
        ('A', 'Ridge_a10_Log'),
    ],
    'Electrical Conductance': [
        ('B', 'Lasso_a0.0010_Log'),
        ('B', 'Elastic_a0.001_l07_Log'),
        ('B', 'XGB_d4_lr003_Log'),
    ],
    'Dissolved Reactive Phosphorus': [
        ('B', 'Ridge_a30_Log'),
        ('B', 'Lasso_a0.0010_Log'),
        ('B', 'XGB_d3_lr003_Log'),
    ],
}

display(pd.DataFrame(
    [(t, fs, m) for t, recipes in TARGET_SWEEP.items() for fs, m in recipes],
    columns=['target', 'feature_set', 'model']
))

## Grouped evaluation helpers

This cell:
- runs grouped out-of-fold predictions,
- reports worst-region behavior,
- saves final artifacts for finalists.

In [ ]:
def grouped_oof_eval(df_local, target, estimator, features_used, allowed_regions=None):
    '''
    Run grouped OOF evaluation with LeaveOneGroupOut.

    Returns:
    - full OOF predictions
    - global metrics
    - mean/min fold R2 across regions
    - fold-by-fold table
    '''
    d = df_local.copy()

    if allowed_regions is not None:
        d = d[d['Region'].isin(allowed_regions)].copy()

    X = d[features_used]
    y = d[target].astype(float).reset_index(drop=True)
    groups = d['Region'].astype(str).reset_index(drop=True)

    logo = LeaveOneGroupOut()

    pipe = Pipeline([
        ('preprocessor', get_preprocessor(features_used)),
        ('model', clone(estimator))
    ])

    pred = cross_val_predict(
        pipe,
        X,
        y,
        groups=groups,
        cv=logo,
        n_jobs=-1,
        verbose=0
    )

    pred = pd.Series(pred).reset_index(drop=True)

    fold_rows = []
    for group_name in groups.unique():
        mask = groups == group_name
        if mask.sum() >= 2:
            fold_rows.append({
                'group': group_name,
                'n': int(mask.sum()),
                'r2': float(r2_score(y[mask], pred[mask])),
            })

    fold_df = pd.DataFrame(fold_rows)

    return {
        'pred': pred.values,
        'rmse': float(np.sqrt(mean_squared_error(y, pred))),
        'mae': float(mean_absolute_error(y, pred)),
        'r2': float(r2_score(y, pred)),
        'mean_fold_r2': float(fold_df['r2'].mean()) if not fold_df.empty else np.nan,
        'min_fold_r2': float(fold_df['r2'].min()) if not fold_df.empty else np.nan,
        'fold_df': fold_df,
        'n_rows': int(len(d)),
        'n_groups': int(groups.nunique()),
    }


def fit_full_and_save(df_local, target, estimator, features_used, run_name):
    '''
    Fit the final full-data pipeline and save preprocessor + model artifacts.
    '''
    X = df_local[features_used]
    y = df_local[target].astype(float)

    pre = get_preprocessor(features_used)
    Xp = pre.fit_transform(X)

    mdl = clone(estimator)
    mdl.fit(Xp, y)

    preproc_path = os.path.join(ARTIFACT_DIR, f'{run_name}__preproc.joblib')
    model_path = os.path.join(ARTIFACT_DIR, f'{run_name}__model.joblib')

    joblib.dump(pre, preproc_path)
    joblib.dump(mdl, model_path)

    return preproc_path, model_path

## Stage 1: Scout run

We first run only a narrow shortlist, optionally prioritizing the hardest regions.

In [ ]:
preferred_scout = ['Eastern_Cape', 'Western_Cape', 'Northern_Bulk']
available_regions = set(df['Region'].astype(str).unique().tolist())
SCOUT_REGIONS = [r for r in preferred_scout if r in available_regions]

if len(SCOUT_REGIONS) < 2:
    SCOUT_REGIONS = None

print('Scout regions:', SCOUT_REGIONS if SCOUT_REGIONS is not None else 'ALL')

rows_scout = []

for target in TARGET_COLS:
    for feature_set_name, model_name in TARGET_SWEEP[target]:
        features = features_for_set(df, feature_set_name)
        estimator = clone(MODEL_BANK[model_name])
        run_name = f'SCOUT__{model_name}__{feature_set_name}__{target_key(target)}'

        print(f'\n--- {run_name} ---')

        with mlflow.start_run(run_name=run_name):
            t0 = time.time()

            out = grouped_oof_eval(
                df_local=df,
                target=target,
                estimator=estimator,
                features_used=features,
                allowed_regions=SCOUT_REGIONS
            )

            dt = time.time() - t0

            mlflow.log_param('stage', 'scout')
            mlflow.log_param('target', target)
            mlflow.log_param('model_name', model_name)
            mlflow.log_param('feature_set_name', feature_set_name)
            mlflow.log_param('split_strategy', SPLIT_STRATEGY)
            mlflow.log_param('group_definition_version', GROUP_DEFINITION_VERSION)
            mlflow.log_param('group_values_hash', compute_group_values_hash(df['Region'].astype(str)))
            mlflow.log_param('feature_set_hash', compute_feature_set_hash(features))
            mlflow.log_param('pipeline_version', PIPELINE_VERSION)
            mlflow.log_param('preprocess_version', PREPROCESS_VERSION)
            mlflow.log_param('n_features_used', len(features))

            mlflow.log_metric('r2', out['r2'])
            mlflow.log_metric('mean_fold_r2', out['mean_fold_r2'])
            mlflow.log_metric('min_fold_r2', out['min_fold_r2'])
            mlflow.log_metric('rmse', out['rmse'])
            mlflow.log_metric('mae', out['mae'])
            mlflow.log_metric('cv_time_sec', dt)

            rows_scout.append({
                'stage': 'scout',
                'run_name': run_name,
                'target': target,
                'model_name': model_name,
                'feature_set': feature_set_name,
                'features_used_json': json.dumps(features),
                'r2': out['r2'],
                'mean_fold_r2': out['mean_fold_r2'],
                'min_fold_r2': out['min_fold_r2'],
                'rmse': out['rmse'],
                'mae': out['mae'],
                'cv_time_sec': dt,
            })

        print(out['fold_df'])
        print(
            f"OOF R2={out['r2']:.4f} | "
            f"mean_fold_r2={out['mean_fold_r2']:.4f} | "
            f"min_fold_r2={out['min_fold_r2']:.4f}"
        )

scout_df = pd.DataFrame(rows_scout).sort_values(
    ['target', 'r2', 'min_fold_r2'],
    ascending=[True, False, False]
).reset_index(drop=True)

print('\nScout results:')
display(scout_df)

## Stage 2: Full finalists

For each target, we take the top 2 scout candidates and run full grouped CV.
We also save inference artifacts for those finalists.

In [ ]:
finalist_df = (
    scout_df.sort_values(['target', 'r2', 'min_fold_r2'], ascending=[True, False, False])
            .groupby('target')
            .head(2)
            .reset_index(drop=True)
)

print('Finalists:')
display(finalist_df[['target', 'feature_set', 'model_name', 'r2', 'min_fold_r2', 'run_name']])

rows_full = []
OOF_PREDS = {}

for _, row in finalist_df.iterrows():
    target = row['target']
    model_name = row['model_name']
    feature_set_name = row['feature_set']
    features = json.loads(row['features_used_json'])
    estimator = clone(MODEL_BANK[model_name])

    run_name = f'FULL__{model_name}__{feature_set_name}__{target_key(target)}'
    print(f'\n=== {run_name} ===')

    with mlflow.start_run(run_name=run_name):
        t0 = time.time()

        out = grouped_oof_eval(
            df_local=df,
            target=target,
            estimator=estimator,
            features_used=features,
            allowed_regions=None
        )

        dt = time.time() - t0

        preproc_path, model_path = fit_full_and_save(
            df_local=df,
            target=target,
            estimator=estimator,
            features_used=features,
            run_name=run_name
        )

        mlflow.log_param('stage', 'full')
        mlflow.log_param('target', target)
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('feature_set_name', feature_set_name)
        mlflow.log_param('split_strategy', SPLIT_STRATEGY)
        mlflow.log_param('group_definition_version', GROUP_DEFINITION_VERSION)
        mlflow.log_param('group_values_hash', compute_group_values_hash(df['Region'].astype(str)))
        mlflow.log_param('feature_set_hash', compute_feature_set_hash(features))
        mlflow.log_param('pipeline_version', PIPELINE_VERSION)
        mlflow.log_param('preprocess_version', PREPROCESS_VERSION)
        mlflow.log_param('n_features_used', len(features))

        mlflow.log_metric('r2', out['r2'])
        mlflow.log_metric('mean_fold_r2', out['mean_fold_r2'])
        mlflow.log_metric('min_fold_r2', out['min_fold_r2'])
        mlflow.log_metric('rmse', out['rmse'])
        mlflow.log_metric('mae', out['mae'])
        mlflow.log_metric('cv_time_sec', dt)

        mlflow.log_artifact(preproc_path, artifact_path='submission_assets')
        mlflow.log_artifact(model_path, artifact_path='submission_assets')

        OOF_PREDS[run_name] = out['pred']

        rows_full.append({
            'stage': 'full',
            'run_name': run_name,
            'target': target,
            'model_name': model_name,
            'feature_set': feature_set_name,
            'features_used_json': json.dumps(features),
            'r2': out['r2'],
            'mean_fold_r2': out['mean_fold_r2'],
            'min_fold_r2': out['min_fold_r2'],
            'rmse': out['rmse'],
            'mae': out['mae'],
            'cv_time_sec': dt,
            'preproc_path': preproc_path,
            'model_path': model_path,
        })

    print(out['fold_df'])
    print(
        f"FULL OOF R2={out['r2']:.4f} | "
        f"mean_fold_r2={out['mean_fold_r2']:.4f} | "
        f"min_fold_r2={out['min_fold_r2']:.4f}"
    )

full_df = pd.DataFrame(rows_full).sort_values(
    ['target', 'r2', 'min_fold_r2'],
    ascending=[True, False, False]
).reset_index(drop=True)

print('\nFull results:')
display(full_df)

## Freeze Manifest A

This chooses one safe anchor model per target from the full finalists.
For DRP, we prefer the safer linear models.

In [ ]:
GATE_R2 = 0.00
manifest_A = {}

for target in TARGET_COLS:
    tdf = full_df[full_df['target'] == target].copy()

    if target == 'Dissolved Reactive Phosphorus':
        tdf = tdf[
            tdf['model_name'].isin([
                'Ridge_a30_Log',
                'Lasso_a0.0010_Log',
                'Elastic_a0.001_l07_Log'
            ])
        ]

    tdf = tdf.sort_values(['r2', 'min_fold_r2'], ascending=[False, False])

    passed = tdf[tdf['r2'] >= GATE_R2]
    chosen = passed.iloc[0] if not passed.empty else tdf.iloc[0]

    manifest_A[target] = chosen.to_dict()

print('=== FROZEN MANIFEST A ===')
print(json.dumps({
    k: {
        'run_name': v['run_name'],
        'model_name': v['model_name'],
        'feature_set': v['feature_set'],
        'r2': float(v['r2']),
        'min_fold_r2': float(v['min_fold_r2'])
    }
    for k, v in manifest_A.items()
}, indent=2))

## Submission helper

This loads the saved artifacts for a chosen manifest entry and generates clipped predictions.

In [ ]:
def predict_from_manifest_entry(entry, df_val_local):
    '''
    Predict from a frozen manifest entry using the saved
    preprocessor and model artifacts.
    '''
    feats = json.loads(entry['features_used_json'])
    pre = joblib.load(entry['preproc_path'])
    mdl = joblib.load(entry['model_path'])

    X = df_val_local[feats]
    pred = mdl.predict(pre.transform(X))
    pred = np.asarray(pred, dtype=float)

    return np.clip(pred, 0, None)

## Build Shot A / B / C

- **Shot A** = safe anchor from Manifest A
- **Shot B** = EC aggressive + DRP safe shrink
- **Shot C** = hedge blend between A and B

In [ ]:
df_val = pd.read_parquet('../data/interim/master_test.parquet')
df_val = engineer_features(df_val)

tpl = pd.read_csv('../data/raw/submission_template.csv')
tpl = make_row_id_template(tpl)

# Shot A: safe anchor
shotA = tpl.copy()
for target in TARGET_COLS:
    shotA[target] = predict_from_manifest_entry(manifest_A[target], df_val)

assert_submission_integrity(shotA, tpl, TARGET_COLS)

# Shot B: EC aggressive + DRP safe
shotB = shotA.copy()

ec_pool = full_df[
    (full_df['target'] == 'Electrical Conductance') &
    (full_df['feature_set'] == 'B')
].sort_values(['r2', 'min_fold_r2'], ascending=[False, False])

if not ec_pool.empty and float(ec_pool.iloc[0]['r2']) > float(manifest_A['Electrical Conductance']['r2']):
    best_ec = ec_pool.iloc[0].to_dict()
    shotB['Electrical Conductance'] = predict_from_manifest_entry(best_ec, df_val)
    print('Shot B EC challenger:', best_ec['run_name'])
else:
    print('Shot B EC kept from Shot A')

drp_pool = full_df[
    (full_df['target'] == 'Dissolved Reactive Phosphorus') &
    (full_df['model_name'].isin(['Ridge_a30_Log', 'Lasso_a0.0010_Log', 'Elastic_a0.001_l07_Log']))
].sort_values(['r2', 'min_fold_r2'], ascending=[False, False])

if not drp_pool.empty:
    best_drp = drp_pool.iloc[0].to_dict()
    drp_pred = predict_from_manifest_entry(best_drp, df_val)
    drp_train_median = float(df['Dissolved Reactive Phosphorus'].median())
    shotB['Dissolved Reactive Phosphorus'] = np.clip(0.6 * drp_pred + 0.4 * drp_train_median, 0, None)
    print('Shot B DRP safe challenger:', best_drp['run_name'])
else:
    print('Shot B DRP kept from Shot A')

assert_submission_integrity(shotB, tpl, TARGET_COLS)

# Shot C: hedge blend
shotC = tpl.copy()
shotC['Total Alkalinity'] = np.clip(0.70 * shotA['Total Alkalinity'] + 0.30 * shotB['Total Alkalinity'], 0, None)
shotC['Electrical Conductance'] = np.clip(0.40 * shotA['Electrical Conductance'] + 0.60 * shotB['Electrical Conductance'], 0, None)
shotC['Dissolved Reactive Phosphorus'] = np.clip(0.70 * shotA['Dissolved Reactive Phosphorus'] + 0.30 * shotB['Dissolved Reactive Phosphorus'], 0, None)

assert_submission_integrity(shotC, tpl, TARGET_COLS)

stamp = datetime.now().strftime('%Y%m%d_%H%M')
pathA = f'../data/submission/submission_{stamp}_A_safe.csv'
pathB = f'../data/submission/submission_{stamp}_B_ec_aggr_drp_safe.csv'
pathC = f'../data/submission/submission_{stamp}_C_blend.csv'

os.makedirs('../data/submission', exist_ok=True)

shotA.drop(columns=['row_id']).to_csv(pathA, index=False)
shotB.drop(columns=['row_id']).to_csv(pathB, index=False)
shotC.drop(columns=['row_id']).to_csv(pathC, index=False)

print('Saved files:')
print('A:', pathA)
print('B:', pathB)
print('C:', pathC)

## Submission diagnostics

This final cell prints simple distribution summaries for the three submission variants.

In [ ]:
print('\nShot A stats')
display(shotA[TARGET_COLS].describe().T[['min', 'mean', 'max']])

print('\nShot B stats')
display(shotB[TARGET_COLS].describe().T[['min', 'mean', 'max']])

print('\nShot C stats')
display(shotC[TARGET_COLS].describe().T[['min', 'mean', 'max']])

tracker = pd.DataFrame([
    {'file': pathA, 'hypothesis': 'Safety anchor: frozen manifest only'},
    {'file': pathB, 'hypothesis': 'EC aggressive from Set B, DRP conservative shrink'},
    {'file': pathC, 'hypothesis': 'Blend hedge between A and B'}
])

print('\nSubmission tracker:')
display(tracker)

print('\nUpload order: A -> B -> C')